# ag_scf2025: Angular Gini and related indices for US net worth (SCF 2025)

Companion notebook to *The Angular Gini* (Sections 4.4 and 4.5, Table 5). It replaces the earlier notebooks `Data_scfp2025.ipynb` and `scf2025_wealth_indices.py`.

**What it computes**

1. Pooled descriptive statistics from the public summary extract (cross-check against published Federal Reserve figures).
2. Shares of families with zero or negative net worth and with negative total income, with standard errors (Section 4.4).
3. Percentiles of net worth and the mean of the bottom quartile, with standard errors (Section 4.4).
4. Gini coefficient, Angular Gini, Palma ratio and GE(1) of net worth without and with a floor at zero, with standard errors (Section 4.5, Table 5).

**Inputs** (folder `input/`; not redistributed, download from https://www.federalreserve.gov/econres/scfindex.htm, 2025 survey):
`SCFP2025.csv` (summary extract, CSV), `scf2025x` (main data set, SAS transport file), `scf2025rw1x` (replicate weights, SAS transport file).

**Outputs** (folder `output/`): `ag_scf2025_descriptive.csv`, `ag_scf2025_shares_se.csv`, `ag_scf2025_percentiles.csv`, `ag_scf2025_indices.csv`, `ag_scf2025_indices.xlsx`.

**Run time**: about 10 minutes with the default of 999 replicates for the index standard errors. Set `N_REP = 20` in the first code cell for a quick test run.

**Method**
- Point estimates are the means of the five implicate-wise estimates.
- Standard errors follow the SCF documentation: total variance = U + 1.2 B, where U is the sampling variance across the 999 bootstrap replicates (first implicate, effective replicate weight WT1B x MM) and B is the variance of the estimates across the five implicates.
- Quantiles are weighted step functions (smallest value whose cumulative weight share is at least p), computed within each implicate and then averaged. The pooled version (all implicates stacked) in step 1 is only a cross-check.

**Requirements**: Python with numpy, pandas and openpyxl (see `requirements.txt` in the repository).

In [ ]:
import os, time
import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.utils import get_column_letter as L

# ---- Configuration -----------------------------------------------------------------
N_REP = 999        # bootstrap replicates for the index standard errors (use e.g. 20 for a quick test run)
N_REP_ALL = 999    # number of replicate weights in the SCF file (fixed)
IN_DIR, OUT_DIR = 'input', 'output'
assert 2 <= N_REP <= N_REP_ALL
os.makedirs(OUT_DIR, exist_ok=True)

def find_file(*names):
    """Look for a data file in input/ and in the working directory."""
    for n in names:
        for d in (IN_DIR, '.'):
            p = os.path.join(d, n)
            if os.path.exists(p):
                return p
    raise FileNotFoundError(f'None of {names} found in ./{IN_DIR}/ or in the working directory {os.getcwd()}')

P_EXTRACT = find_file('SCFP2025.csv', 'scfp2025.csv')
P_MAIN = find_file('scf2025x')
P_RW = find_file('scf2025rw1x')
print('Input files:', P_EXTRACT, P_MAIN, P_RW)

In [ ]:
# ---- 1. Load data -------------------------------------------------------------------
ex = pd.read_csv(P_EXTRACT)
ex.columns = [c.upper() for c in ex.columns]
for col in ['WGT', 'NETWORTH', 'INCOME', 'NWCAT', 'Y1', 'YY1']:
    assert col in ex.columns, f'Column {col} is missing in the summary extract'

def read_xport(path, keep=None):
    chunks = pd.read_sas(path, format='xport', chunksize=2000, encoding='latin1')
    return pd.concat([c if keep is None else c[keep] for c in chunks], ignore_index=True)

main = read_xport(P_MAIN, ['YY1', 'Y1', 'X42001', 'X5729'])   # main data set: weight and total income
rw = read_xport(P_RW)                                           # replicate weights

assert (main['Y1'].to_numpy() == ex['Y1'].to_numpy()).all(), 'Row order of summary extract and main data set differ'
nw = ex['NETWORTH'].to_numpy(float)               # net worth (2025 dollars)
inc = ex['INCOME'].to_numpy(float)                # total income in the summary extract
wm = main['X42001'].to_numpy(float)               # weight within each implicate
imp = (main['Y1'] % 10).to_numpy()                # implicate number 1..5
i1 = np.where(imp == 1)[0]                        # rows of the first implicate
assert (main['YY1'].to_numpy()[i1] == rw['YY1'].to_numpy()).all(), 'Replicate weights do not match the first implicate'

# effective replicate weights = WT1B x MM (rows: families of the first implicate)
Wrep = np.stack([rw[f'WT1B{i}'].fillna(0).to_numpy() * rw[f'MM{i}'].fillna(0).to_numpy() for i in range(1, N_REP_ALL + 1)], axis=1)
x = main['X5729'].to_numpy(float).copy()
x[np.abs(x) < 1e-6] = 0                            # SAS missing/zero artefact
print(f'Rows: {len(ex):,} | families: {ex["YY1"].nunique():,} | implicates: {sorted(int(i) for i in np.unique(imp))}')
print(f'Sum of weights WGT: {ex["WGT"].sum()/1e6:,.2f} million families')
print('Weight X42001 / WGT:', np.unique((wm / ex["WGT"].to_numpy(float)).round(6)))
print(f'Sum of replicate weights (mean): {Wrep.sum(0).mean()/1e6:,.2f} million')

In [ ]:
# ---- 2. Index functions -------------------------------------------------------------
def _ang(a, b):
    """Angular differential (2/pi) |atan2(a,b) - atan2(b,a)| for arrays a, b (also for negative values)."""
    th = np.arctan2(a, b); ph = np.pi / 2 - th              # atan2(b,a) = pi/2 - atan2(a,b) (mod 2 pi)
    ph = np.where(ph > np.pi, ph - 2 * np.pi, ph)
    return (2 / np.pi) * np.abs(th - ph)

def gini_ag(y, w, chunk=1200):
    """Weighted Gini coefficient and Angular Gini (all pairs, computed in blocks)."""
    y = np.asarray(y, float); w = np.asarray(w, float); n = len(y); W = w.sum(); mu = (w * y).sum() / W
    sg = sa = 0.0
    for i0 in range(0, n, chunk):
        i1_ = min(i0 + chunk, n); a = y[i0:i1_, None]; wa = w[i0:i1_, None]
        b = y[None, i0:i1_]; ww = wa * w[None, i0:i1_]; d = np.abs(a - b)
        sg += (ww * d).sum(); sa += (ww * d * _ang(a, b)).sum()
        if i1_ < n:
            b = y[None, i1_:]; ww = wa * w[None, i1_:]; d = np.abs(a - b)
            sg += 2 * (ww * d).sum(); sa += 2 * (ww * d * _ang(a, b)).sum()
    den = 2 * W**2 * mu
    return sg / den, sa / den

def palma_w(y, w):
    """Palma ratio: share of the richest 10 % / share of the poorest 40 % (Lorenz curve linearly interpolated)."""
    o = np.argsort(y); y = y[o]; w = w[o]
    p = np.concatenate([[0], np.cumsum(w)]) / w.sum(); Lc = np.concatenate([[0], np.cumsum(w * y)]) / (w * y).sum()
    return (1 - np.interp(0.9, p, Lc)) / np.interp(0.4, p, Lc)

def ge1_w(y, w):
    """Generalized entropy index GE(1) (Theil T); not defined (NaN) for negative values."""
    if (y < 0).any():
        return np.nan
    r = y / ((w * y).sum() / w.sum()); t = np.where(r > 0, r * np.log(np.where(r > 0, r, 1.0)), 0.0)
    return (w * t).sum() / w.sum()

def four(y, w):
    g, a = gini_ag(y, w)
    return np.array([g, a, palma_w(y, w), ge1_w(y, w)])

def stats8(y, w):
    """[Gini, AG, Palma, GE1] without floor, followed by the same four indices with negative values set to 0."""
    return np.concatenate([four(y, w), four(np.clip(y, 0, None), w)])

# ---- Benchmark: data (-4, 5, 0) must give Gini 6.00 and Angular Gini 11.24 (Section 4.1) ----
g, a = gini_ag([-4, 5, 0], np.ones(3))
assert abs(g - 6.0) < 5e-3 and abs(a - 11.24) < 5e-3, 'Benchmark (-4, 5, 0) failed'
bench = {'no_floor': (g, a), 'floor': gini_ag([0, 5, 0], np.ones(3))}
print(f'Benchmark (-4, 5, 0): Gini {g:.2f}, Angular Gini {a:.2f}: passed')

In [ ]:
# ---- 3. Pooled descriptive statistics (summary extract, weights WGT, implicates stacked) ----
w_all = ex['WGT'].to_numpy(float); total_w = w_all.sum()

def share(mask):
    """Weighted share (percent) of families for which mask is true."""
    return 100 * w_all[mask].sum() / total_w

_order = np.argsort(nw); _nw_sorted = nw[_order]; _cum = np.cumsum(w_all[_order]) / total_w
def wquantile(p):
    """Pooled weighted quantile as a step function: smallest value whose cumulative weight share is >= p."""
    return _nw_sorted[np.searchsorted(_cum, p, side='left')]

mean_all = np.average(nw, weights=w_all)
print(f'Mean net worth:   {mean_all:>14,.0f} $')
print(f'Median net worth: {wquantile(0.5):>14,.0f} $\n')

rows = []
for t in [0, 100, 1000, 5000]:
    rows.append({'Threshold ($)': t, 'Share NW <= threshold (%)': share(nw <= t),
                 'of which above zero (0 < NW <= threshold, %)': share((nw > 0) & (nw <= t))})
print(pd.DataFrame(rows).round(2).to_string(index=False))
print(f'\nUnweighted rows with NW == 0: {(nw == 0).sum()} of {len(nw):,}')

print(f'\nMinimum of INCOME in the extract: {inc.min():,.2f}')
print(f'INCOME == 0: {share(inc == 0):.3f} % ({(inc == 0).sum()} rows)')
print('Note: in the summary extract INCOME has no negative values. In the public main data set, negative total income')
print('(X5729) is coded -9; its share is therefore estimated from the main data set below.\n')

pct = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 25, 50]
print('Pooled percentiles of net worth (cross-check only; the manuscript uses implicate-wise values, see below):')
print(pd.DataFrame({'Percentile': pct, 'Net worth ($)': [wquantile(p / 100) for p in pct]}).to_string(
    index=False, formatters={'Net worth ($)': '{:,.0f}'.format}))

# Bottom quartile, pooled, using the Federal Reserve percentile group NWCAT == 1, against published figures
bq = ex['NWCAT'].to_numpy() == 1
bq_mean = np.average(nw[bq], weights=w_all[bq])
cmp_ = pd.DataFrame({'Microdata (here, pooled)': [bq_mean / 1e3, mean_all / 1e3],
                     'Chartbook CSV (published)': [-3.93, 1232.0], 'Report (published)': [-4.3, 1241.5]},
                    index=['Mean, bottom quartile (thousand $)', 'Mean, all families (thousand $)']).round(2)
print(f'\nBottom quartile (NWCAT == 1): share {share(bq):.2f} %')
print(cmp_.to_string())

# Quantities used in the Excel workbook (pooled)
neg = nw < 0
desc = {'mean_raw': mean_all, 'mean_floor': (w_all * np.clip(nw, 0, None)).sum() / total_w,
        'debt_share': 100 * (w_all[neg] * -nw[neg]).sum() / (w_all * nw).sum(),
        'mean_debt': -(w_all[neg] * nw[neg]).sum() / w_all[neg].sum(), 'min': nw.min()}

res = {'Families (million, sum of WGT)': total_w / 1e6,
       'Share NW < 0 (%)': share(nw < 0), 'Share NW == 0 (%)': share(nw == 0), 'Share NW <= 0 (%)': share(nw <= 0),
       'Share NW <= $100 (%)': share(nw <= 100), 'Share NW <= $1,000 (%)': share(nw <= 1000),
       'Share INCOME == 0 (%)': share(inc == 0), 'Minimum INCOME ($)': inc.min(),
       'Percentile 1 NW, pooled ($)': wquantile(0.01), 'Percentile 7 NW, pooled ($)': wquantile(0.07),
       'Percentile 8 NW, pooled ($)': wquantile(0.08),
       'Mean bottom quartile NW, pooled, NWCAT == 1 ($)': bq_mean, 'Mean NW, all families ($)': mean_all,
       'Median NW, pooled ($)': wquantile(0.5)}
out = pd.Series(res, name='SCF 2025').round(3)
out.to_csv(os.path.join(OUT_DIR, 'ag_scf2025_descriptive.csv'), header=True)

In [ ]:
# ---- 4. Shares with standard errors (replicate weights and implicates) -------------------
stats = {
    'Net worth < 0':                          nw < 0,
    'Net worth = 0':                          nw == 0,
    'Net worth <= 0':                         nw <= 0,
    'Net worth <= $100':                      nw <= 100,
    'Net worth <= $1,000':                    nw <= 1000,
    'Total income negative (X5729 = -9)':     x == -9,
    'Total income zero (X5729 = -1)':         x == -1,
    'Total income negative or zero':          (x == -9) | (x == -1),
    'INCOME == 0 (summary extract)':          inc == 0,
}
rows = []
for name, mask in stats.items():
    ind = mask.astype(float)
    est_k = np.array([100 * (wm[imp == k] * ind[imp == k]).sum() / wm[imp == k].sum() for k in range(1, 6)])
    U = (100 * (Wrep * ind[i1][:, None]).sum(0) / Wrep.sum(0)).var(ddof=1)     # sampling variance (replicates)
    B = est_k.var(ddof=1)                                                       # imputation variance
    rows.append({'Measure': name, 'Share (%)': est_k.mean(), 'SE total': np.sqrt(U + 1.2 * B),
                 'SE sampling': np.sqrt(U), 'SE imputation': np.sqrt(1.2 * B)})
shares_se = pd.DataFrame(rows).set_index('Measure')
shares_se.to_csv(os.path.join(OUT_DIR, 'ag_scf2025_shares_se.csv'))
print(shares_se.round(3).to_string())
n9 = x == -9
print(f'\nRows with X5729 = -9: {n9.sum()} (families: {main.loc[n9, "YY1"].nunique()}); of these INCOME == 0 in the extract: '
      f'{(inc[n9] == 0).sum()}, INCOME > 0: {(inc[n9] > 0).sum()}')

In [ ]:
# ---- 5. Percentiles and mean of the bottom quartile (implicate-wise, 999 replicates) -------
pos = [np.where(Wrep[:, b] > 0)[0] for b in range(N_REP_ALL)]     # families with positive replicate weight

def wq(y, w, p):
    """Weighted quantile as a step function (smallest value whose cumulative weight share is >= p)."""
    o = np.argsort(y); c = np.cumsum(w[o]) / w.sum()
    return y[o][np.searchsorted(c, p, side='left')]

def pstats(y, w):
    """[1st percentile, 8th percentile, mean of the bottom quartile, 25th percentile]."""
    p25 = wq(y, w, 0.25); mk = y <= p25
    return np.array([wq(y, w, 0.01), wq(y, w, 0.08), np.average(y[mk], weights=w[mk]), p25])

pest = np.array([pstats(nw[imp == k], wm[imp == k]) for k in range(1, 6)])
prep = np.array([pstats(nw[i1][pos[b]], Wrep[pos[b], b]) for b in range(N_REP_ALL)])
pth = pest.mean(0)
pse = np.sqrt(prep.var(0, ddof=1) + 1.2 * pest.var(0, ddof=1))
pct_labels = ['1st percentile of net worth ($)', '8th percentile of net worth ($)',
              'Mean of the bottom quartile ($)', '25th percentile of net worth ($)']
pct_rows = list(zip(pct_labels, pth, pse))
pd.DataFrame({'Estimate': pth, 'SE': pse}, index=pct_labels).to_csv(os.path.join(OUT_DIR, 'ag_scf2025_percentiles.csv'))
print(pd.DataFrame({'Estimate': pth, 'SE': pse}, index=pct_labels).round(1).to_string())

In [ ]:
# ---- 6. Indices without and with a floor at zero, standard errors (replicates + implicates) --
est = np.array([stats8(nw[imp == k], wm[imp == k]) for k in range(1, 6)])

t0 = time.time()
for b in range(min(5, N_REP)):
    stats8(nw[i1][pos[b]], Wrep[pos[b], b])
proj = (time.time() - t0) / min(5, N_REP) * N_REP
print(f'Estimated run time for {N_REP} replicates: {proj:.0f} s')
t0 = time.time()
rep = np.array([stats8(nw[i1][pos[b]], Wrep[pos[b], b]) for b in range(N_REP)])
elapsed = time.time() - t0
print(f'Done: {N_REP} replicates in {elapsed:.0f} s')

def derived(v):
    G0, A0, P0, E0, G1, A1, P1, E1 = v
    return np.array([G0, A0, P0, E0, G1, A1, P1, E1, G1 - G0, A1 - A0, P1 - P0, A0 / G0, A1 / G1,
                     100 * (G1 / G0 - 1), 100 * (A1 / A0 - 1), 100 * (P1 / P0 - 1)])

base = ['Gini', 'AG', 'Palma', 'GE1']
names = ([f'{b} (no floor)' for b in base] + [f'{b} (floor at 0)' for b in base] +
         ['Change Gini', 'Change AG', 'Change Palma', 'AG/Gini (no floor)', 'AG/Gini (floor at 0)',
          'Rel. change Gini (%)', 'Rel. change AG (%)', 'Rel. change Palma (%)'])
Fe = np.array([derived(v) for v in est]); Fr = np.array([derived(v) for v in rep])
theta = Fe.mean(0); B_imp = Fe.var(0, ddof=1)
with np.errstate(all='ignore'):
    U_rep = np.nanvar(Fr, axis=0, ddof=1)
se = np.sqrt(U_rep + 1.2 * B_imp)
R = pd.DataFrame({'Estimate': theta, 'SE': se}, index=names)
assert R.loc['AG (floor at 0)', 'Estimate'] <= R.loc['Gini (floor at 0)', 'Estimate'], 'AG > Gini for non-negative data'
R.to_csv(os.path.join(OUT_DIR, 'ag_scf2025_indices.csv'))
print(R.round(4).to_string())

In [ ]:
# ---- 7. Excel workbook ----------------------------------------------------------------------
F = lambda **k: Font(name='Arial', **{'size': 10, **k})
BLUE, BLACK, HDR = F(color='0000FF'), F(), F(bold=True, color='FFFFFF'); HF = PatternFill('solid', start_color='305496')
wrap = Alignment(wrap_text=True, vertical='top'); ctr = Alignment(horizontal='center', vertical='center', wrap_text=True)
def header(ws, row, labels):
    for j, t in enumerate(labels, 1):
        c = ws.cell(row=row, column=j, value=t); c.font = HDR; c.fill = HF; c.alignment = ctr
    ws.row_dimensions[row].height = 34
v = lambda x: None if pd.isna(x) else float(x)

wb = Workbook(); ws = wb.active; ws.title = 'Info'
info = [('SCF 2025: net worth of all US families, indices without and with a floor at 0', True), ('', False),
 ('Blue = computed input value (from the notebook ag_scf2025.ipynb), black = Excel formula. Colour scales are conditional formats.', False), ('', False),
 ('Data: summary extract SCFP2025.csv (NETWORTH, 2025 dollars), main data set scf2025x (weight X42001), replicate weights scf2025rw1x. 4,360 families in 5 implicates, sum of weights %.2f million families.' % (total_w / 1e6), False),
 ('Variants: "no floor" = net worth as surveyed (negative values kept); "floor at 0" = negative values set to 0.', False),
 ('Indices (weighted): Gini; Angular Gini (angle formula with atan2, can also be evaluated for negative values as long as the mean is positive); Palma = share of the richest 10 % / share of the poorest 40 % (Lorenz curve linearly interpolated); GE(1) = Theil T, not defined for negative values (n/d).', False),
 ('Estimator: mean of the five implicate estimates. Standard error: sampling variance from bootstrap replicates (first implicate, weight WT1B x MM) plus imputation variance across the implicates, total variance = U + (1 + 1/5) B; differences and ratios have their own standard errors from the same replicates.', False),
 (f'Replicates: {N_REP} for the indices (estimated run time {proj:.0f} s) and 999 for the percentiles. With fewer replicates the sampling variance is estimated less precisely.', False), ('', False),
 ('Sheets: Indices | Descriptive (incl. percentiles, 999 replicates) | Implicates | Benchmark', True)]
for i, (t, b) in enumerate(info, 1):
    c = ws.cell(row=i, column=1, value=t); c.font = F(bold=b, size=12 if i == 1 else 10); c.alignment = wrap
ws.column_dimensions['A'].width = 130

ws = wb.create_sheet('Indices')
header(ws, 1, ['Index', 'No floor: estimate', 'No floor: SE', 'Floor at 0: estimate', 'Floor at 0: SE',
               'Change due to floor', 'SE of change', 'Relative change (%)'])
rows_x = [('Gini', 'Gini (no floor)', 'Gini (floor at 0)', 'Change Gini'), ('Angular Gini', 'AG (no floor)', 'AG (floor at 0)', 'Change AG'),
          ('Palma', 'Palma (no floor)', 'Palma (floor at 0)', 'Change Palma'), ('GE(1)', 'GE1 (no floor)', 'GE1 (floor at 0)', None)]
for r, (lab, a_, b_, d_) in enumerate(rows_x, 2):
    ws.cell(row=r, column=1, value=lab).font = BLACK
    for col, key, k in [(2, a_, 'Estimate'), (3, a_, 'SE'), (4, b_, 'Estimate'), (5, b_, 'SE')]:
        xv = v(R.loc[key, k]); c = ws.cell(row=r, column=col, value=('n/d' if xv is None else round(xv, 5))); c.font = BLUE
        c.number_format = '0.000' if lab != 'Palma' else '0.0'; c.alignment = Alignment(horizontal='right')
    ws.cell(row=r, column=6, value=f'=IF(ISNUMBER(B{r}),D{r}-B{r},"n/d")').font = BLACK
    sed = v(R.loc[d_, 'SE']) if d_ else None
    c = ws.cell(row=r, column=7, value=('n/d' if sed is None else round(sed, 5))); c.font = BLUE; c.alignment = Alignment(horizontal='right')
    ws.cell(row=r, column=8, value=f'=IF(ISNUMBER(B{r}),100*(D{r}/B{r}-1),"n/d")').font = BLACK
    for col in (6, 7): ws.cell(row=r, column=col).number_format = '0.0000' if lab != 'Palma' else '0.0'
    ws.cell(row=r, column=8).number_format = '0.00'
r = 6
ws.cell(row=r, column=1, value='Angular Gini / Gini').font = BLACK
for col, key, k in [(2, 'AG/Gini (no floor)', 'Estimate'), (3, 'AG/Gini (no floor)', 'SE'), (4, 'AG/Gini (floor at 0)', 'Estimate'), (5, 'AG/Gini (floor at 0)', 'SE')]:
    c = ws.cell(row=r, column=col, value=round(float(R.loc[key, k]), 5)); c.font = BLUE; c.number_format = '0.000'
ws.cell(row=r, column=6, value='=D6-B6').font = BLACK; ws.cell(row=r, column=6).number_format = '0.0000'
ws.cell(row=r, column=8, value='=100*(D6/B6-1)').font = BLACK; ws.cell(row=r, column=8).number_format = '0.00'
ws.conditional_formatting.add('H2:H6', ColorScaleRule(start_type='min', start_color='F8A5A5', mid_type='num', mid_value=0, mid_color='FFFFFF', end_type='max', end_color='9BC2E6'))
ws.cell(row=8, column=1, value='Reading aid: standard errors are in the SE columns. The change due to the floor is small for the Gini coefficient and the Angular Gini and large for the Palma ratio (the poorest 40 % hold only a small share of wealth). n/d = not defined without the floor (logarithm of negative values).').font = F(italic=True)
ws.column_dimensions['A'].width = 22
for j in range(2, 9): ws.column_dimensions[L(j)].width = 17
ws.freeze_panes = 'B2'

ws = wb.create_sheet('Descriptive')
header(ws, 1, ['Measure', 'Value', 'SE'])
for r, (lab, row_) in enumerate(shares_se.iterrows(), 2):
    ws.cell(row=r, column=1, value=lab + ' (% of families)').font = BLACK
    c = ws.cell(row=r, column=2, value=round(float(row_['Share (%)']), 4)); c.font = BLUE; c.number_format = '0.00'
    c = ws.cell(row=r, column=3, value=round(float(row_['SE total']), 4)); c.font = BLUE; c.number_format = '0.00'
base_r = len(shares_se) + 2
items = [('Mean net worth without floor ($)', desc['mean_raw'], '#,##0'), ('Mean net worth with floor ($)', desc['mean_floor'], '#,##0')]
for k, (lab, xv, fmt) in enumerate(items):
    ws.cell(row=base_r + k, column=1, value=lab).font = BLACK; c = ws.cell(row=base_r + k, column=2, value=round(float(xv), 2)); c.font = BLUE; c.number_format = fmt
ws.cell(row=base_r + 2, column=1, value='Increase of the mean due to the floor (%)').font = BLACK
c = ws.cell(row=base_r + 2, column=2, value=f'=100*(B{base_r+1}/B{base_r}-1)'); c.font = BLACK; c.number_format = '0.000'
for k, (lab, xv, fmt) in enumerate([('Total debt as % of total net worth', desc['debt_share'], '0.000'),
                                    ('Mean debt of families with negative net worth ($)', desc['mean_debt'], '#,##0'),
                                    ('Minimum net worth in the public data set ($)', desc['min'], '#,##0')]):
    ws.cell(row=base_r + 3 + k, column=1, value=lab).font = BLACK; c = ws.cell(row=base_r + 3 + k, column=2, value=round(float(xv), 3)); c.font = BLUE; c.number_format = fmt
for k, (lab, xv, e_) in enumerate(pct_rows):
    r = base_r + 6 + k
    ws.cell(row=r, column=1, value=lab + ' (999 replicates)').font = BLACK
    c = ws.cell(row=r, column=2, value=round(float(xv), 1)); c.font = BLUE; c.number_format = '#,##0'
    c = ws.cell(row=r, column=3, value=round(float(e_), 1)); c.font = BLUE; c.number_format = '#,##0'
ws.column_dimensions['A'].width = 68; ws.column_dimensions['B'].width = 16; ws.column_dimensions['C'].width = 10

ws = wb.create_sheet('Implicates')
header(ws, 1, ['Implicate'] + [f'{n} ({t})' for t in ('no floor', 'floor at 0') for n in ('Gini', 'Angular Gini', 'Palma', 'GE(1)')])
for k in range(5):
    ws.cell(row=2 + k, column=1, value=k + 1).font = BLACK
    for j in range(8):
        xv = v(est[k, j]); c = ws.cell(row=2 + k, column=2 + j, value=('n/d' if xv is None else round(xv, 5))); c.font = BLUE; c.number_format = '0.0000'; c.alignment = Alignment(horizontal='right')
ws.cell(row=7, column=1, value='Mean').font = BLACK
for j in range(8):
    col = L(2 + j); c = ws.cell(row=7, column=2 + j, value=f'=IF(COUNT({col}2:{col}6)=5,AVERAGE({col}2:{col}6),"n/d")'); c.font = BLACK; c.number_format = '0.0000'
for j in range(1, 10): ws.column_dimensions[L(j)].width = 17

ws = wb.create_sheet('Benchmark')
header(ws, 1, ['Check', 'Gini', 'Angular Gini', 'Expected (manuscript)'])
ws.append(['Data (-4, 5, 0), no floor', round(float(bench['no_floor'][0]), 4), round(float(bench['no_floor'][1]), 4), 'Gini 6.00; Angular Gini 11.24'])
ws.append(['Data (0, 5, 0), floor at 0', round(float(bench['floor'][0]), 4), round(float(bench['floor'][1]), 4), 'Angular Gini not above the Gini coefficient (bound)'])
for row in ws.iter_rows(min_row=2, max_row=3):
    for c in row: c.font = BLUE if c.column in (2, 3) else BLACK
ws.column_dimensions['A'].width = 38
for j in range(2, 5): ws.column_dimensions[L(j)].width = 28
wb.save(os.path.join(OUT_DIR, 'ag_scf2025_indices.xlsx'))
print('Saved:', ', '.join(sorted(os.listdir(OUT_DIR))))